# Imports

In [51]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

# Load engineered dataset

In [41]:
project_root = Path.cwd().parent
features_path = project_root / "data" / "spy_features.csv"

df = pd.read_csv(features_path)
df["Date"] = pd.to_datetime(df["Date"])

df.head()

,Date,Close,High,Low,Open,Volume,target,return_1d,return_5d,ma_5_rel,...,vol_ratio,volume_change_1d,range_pos_10,range_pos_20,ma_spread_5_20,ma_spread_10_20,intraday_return,high_low_range,close_to_high,close_to_low
0,2010-02-02,82.814667,82.972223,81.689264,81.974369,216327900,0,0.012103,0.009789,0.012233,...,1.195418,0.151507,0.437069,0.398994,-0.027178,-0.019462,0.010251,0.015705,-0.001899,0.013777
1,2010-02-03,82.402031,82.889707,82.161945,82.439541,172730700,0,-0.004983,0.000000,0.007190,...,1.217450,-0.201533,0.370218,0.329552,-0.025527,-0.021425,-0.000455,0.008858,-0.005883,0.002922
2,2010-02-04,79.858604,81.801798,79.843596,81.764288,356715700,1,-0.030866,-0.019618,-0.020070,...,1.514461,1.065155,0.003760,0.002294,-0.026170,-0.022952,-0.023307,0.024525,-0.023755,0.000188
3,2010-02-05,80.023659,80.188713,78.463099,79.948627,493585800,0,0.002067,-0.006797,-0.016723,...,1.495205,0.383695,0.346089,0.196970,-0.024185,-0.021936,0.000938,0.021993,-0.002058,0.019889
4,2010-02-08,79.445938,80.526327,79.385915,80.083665,224166900,1,-0.007219,-0.029067,-0.018083,...,1.297265,-0.545840,0.217967,0.124052,-0.026086,-0.021607,-0.007963,0.014365,-0.013417,0.000756


# Define features and target

In [42]:
feature_cols = [
    "return_1d",
    "return_5d",
    "mom_2d",
    "mom_10d",
    "mom_20d",
    "ma_5_rel",
    "ma_10_rel",
    "ma_20_rel",
    "ma_spread_5_20",
    "ma_spread_10_20",
    "vol_5",
    "vol_20",
    "vol_ratio",
    "volume_change_1d",
    "range_pos_10",
    "range_pos_20",
    "intraday_return",
    "high_low_range",
    "close_to_high",
    "close_to_low"
]

X = df[feature_cols]
y = df["target"]

# Chronological train / validation / test split

In [43]:
n = len(df)

train_end = int(0.7 * n)
val_end = int(0.85 * n)

X_train = X.iloc[:train_end]
y_train = y.iloc[:train_end]

X_val = X.iloc[train_end:val_end]
y_val = y.iloc[train_end:val_end]

X_test = X.iloc[val_end:]
y_test = y.iloc[val_end:]

# Train logistic regression with feature scaling

In [44]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)

val_pred = model.predict(X_val)
print("Validation accuracy:", accuracy_score(y_val, val_pred))

Validation accuracy: 0.5


# Evaluate classification performance

In [50]:
val_proba = model.predict_proba(X_val)[:, 1]

print("Validation accuracy:", accuracy_score(y_val, val_pred))
print("Validation AUC:", roc_auc_score(y_val, val_proba))
print("Fraction predicted up:", val_pred.mean())
print("Probability min/max/mean:", val_proba.min(), val_proba.max(), val_proba.mean())
print(classification_report(y_val, val_pred))
print(confusion_matrix(y_val, val_pred))

Validation accuracy: 0.5
Validation AUC: 0.49293762379452544
Fraction predicted up: 0.9325657894736842
Probability min/max/mean: 0.3739506738188563 0.7178705234145705 0.5582814532112037
              precision    recall  f1-score   support

           0       0.44      0.06      0.11       299
           1       0.50      0.93      0.65       309

    accuracy                           0.50       608
   macro avg       0.47      0.49      0.38       608
weighted avg       0.47      0.50      0.38       608

[[ 18 281]
 [ 23 286]]


# Train a nonlinear baseline: random forest

In [53]:
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=5,
    min_samples_leaf=10,
    random_state=42
)

model.fit(X_train, y_train)

val_pred = model.predict(X_val)

Validation accuracy: 0.5164473684210527
Validation AUC: 0.4806853481399703
Fraction predicted up: 0.9391447368421053
              precision    recall  f1-score   support

           0       0.57      0.07      0.12       299
           1       0.51      0.95      0.67       309

    accuracy                           0.52       608
   macro avg       0.54      0.51      0.40       608
weighted avg       0.54      0.52      0.40       608

[[ 21 278]
 [ 16 293]]


# Evaluate classification performance

In [ ]:
val_proba = model.predict_proba(X_val)[:, 1]

print("Validation accuracy:", accuracy_score(y_val, val_pred))
print("Validation AUC:", roc_auc_score(y_val, val_proba))
print("Fraction predicted up:", val_pred.mean())
print("Probability min/max/mean:", val_proba.min(), val_proba.max(), val_proba.mean())
print(classification_report(y_val, val_pred))
print(confusion_matrix(y_val, val_pred))

# Results

In [56]:
results = pd.DataFrame([
    {
        "model": "logreg_scaled",
        "target_horizon_days": 1,
        "val_accuracy": 0.5000,
        "val_auc": 0.4929,
        "predicted_up_fraction": 0.9326
    },
    {
        "model": "random_forest",
        "target_horizon_days": 1,
        "val_accuracy": 0.5164,
        "val_auc": 0.4807,
        "predicted_up_fraction": 0.9391
    }
])

results_path = project_root / "results" / "model_comparison.csv"
results.to_csv(results_path, index=False)